# Task 2 - Incremental CPG Parser Service

The service uses Python `ast`, stable structural IDs, statement-level CFG, lexical reaching-definitions DFG, and same-file call resolution. It releases each file graph before processing the next file.

In [1]:
import json
import sys
from collections import Counter
from pathlib import Path

root = Path('..').resolve()
sys.path.insert(0, str(root))
from cpg_parser.analyzer import CPGAnalyzer
from cpg_parser.discovery import discover_repo
from cpg_parser.ids import file_id

repo = root / 'source-repo'
report = discover_repo(repo)
node_counts, edge_counts = Counter(), Counter()
node_ids, edge_ids = set(), set()
warnings = 0
for relative in report.files:
    result = CPGAnalyzer(
        (repo / relative).read_text(encoding='utf-8', errors='replace'),
        file_id('huggingface/optimum', relative), relative,
    ).analyze()
    node_counts.update(result.node_counts())
    edge_counts.update(result.edge_counts())
    node_ids.update(node.id for node in result.nodes)
    edge_ids.update(edge.id for edge in result.edges)
    warnings += len(result.warnings)
summary = {
    'repository': 'huggingface/optimum',
    'files': len(report.files),
    'nodes': sum(node_counts.values()),
    'edges': sum(edge_counts.values()),
    'node_counts': dict(sorted(node_counts.items())),
    'edge_counts': dict(sorted(edge_counts.items())),
    'warnings': warnings,
    'all_node_ids_unique': len(node_ids) == sum(node_counts.values()),
    'all_edge_ids_unique': len(edge_ids) == sum(edge_counts.values()),
}
print(json.dumps(summary, indent=2))
assert {'AST', 'CFG', 'DFG', 'CALL'} <= set(edge_counts)
assert summary['all_node_ids_unique'] and summary['all_edge_ids_unique']
print('PASS: all CPG categories exist and IDs are unique')

{
  "repository": "huggingface/optimum",
  "files": 61,
  "nodes": 62397,
  "edges": 77849,
  "node_counts": {
    "AST": 58021,
    "EXTERNAL": 2894,
    "SYNTHETIC": 1482
  },
  "edge_counts": {
    "AST": 57960,
    "CALL": 2593,
    "CFG": 7248,
    "DFG": 10048
  },
  "warnings": 31,
  "all_node_ids_unique": true,
  "all_edge_ids_unique": true
}
PASS: all CPG categories exist and IDs are unique


In [2]:
import subprocess
import sys
from pathlib import Path

result = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q'],
    cwd=Path('..').resolve(), capture_output=True, text=True, check=True,
)
print(result.stdout.rstrip())
assert '[100%]' in result.stdout
print('PASS: parser, replay, schema, and syntax-error tests')

.............                                                            [100%]
PASS: parser, replay, schema, and syntax-error tests


## Reflection

The analyzer is intentionally conservative for aliases, dynamic dispatch, container mutation, and exact exception flow. Tests verify deterministic IDs, graph categories, stale deletion, schema contracts, and safe syntax-error routing.